# Bodrum Hotel & Destination Intelligence
## 04 - Temel Feature Engineering

Bu notebook, temizlenmiş tek ana `df` tablosuna EDA öncesinde kullanılabilecek temel ve açıklanabilir özellikleri ekler.

Kapsam bilinçli olarak sınırlıdır:

- Eksik fiyat, resmî yıldız veya diğer alanlar doldurulmaz.
- `value_score`, `luxury_score` ve `destination_score` gibi ağırlık seçimi gerektiren bileşik skorlar üretilmez.
- Bölgesel özellikler yalnızca mevcut ana tablodan türetilir; ayrı `area_summary` girdisi kullanılmaz.
- Fiyat özellikleri, sabit otel fiyatı değil, yalnızca 2026-08-24 tarihli snapshot bağlamını temsil eder.


### 1. Kurulum ve temiz verinin yüklenmesi

Girdi `03_data_cleaning.ipynb` tarafından üretilen `data/processed/hotels_clean.csv` dosyasıdır. Notebook boyunca tek ana çalışma tablosu `df` olarak adlandırılır.


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from bodrum_intelligence.features import build_basic_features, save_feature_outputs

CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "hotels_clean.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

assert CLEAN_PATH.exists(), f'Temiz veri bulunamadı: {CLEAN_PATH}'

df = pd.read_csv(CLEAN_PATH, dtype={"phone": "string"})

### 2. Girdi bütünlüğü

Feature engineering başlamadan önce temiz tablonun satır, anahtar ve bölge sayısı bütünlüğü doğrulanır. Bu aşamada satır silinmez.


In [2]:
input_summary = pd.DataFrame(
    {
        "metric": ["row_count", "column_count", "unique_hotel_id", "unique_place_id", "area_count"],
        "value": [
            len(df),
            df.shape[1],
            df["hotel_id"].nunique(),
            df["place_id"].nunique(),
            df["area"].nunique(),
        ],
    }
)
display(input_summary)

expected_area_count = df.groupby("area")["hotel_id"].transform("size")
assert df["area_hotel_count"].eq(expected_area_count).all(), 'area_hotel_count ana tabloyla eşleşmiyor.'

,metric,value
0,row_count,192
1,column_count,20
2,unique_hotel_id,192
3,unique_place_id,192
4,area_count,14


### 3. Temel özelliklerin üretilmesi

Özellikler dört grupta hazırlanır:

1. Bilgi varlık göstergeleri
2. Yorum hacmi ve şeffaf ağırlıklı puan
3. Bölge medyanına göre puan farkı
4. Bölge içi fiyat ve yorum konumu

Orijinal kolonlar değiştirilmez; yeni kolonlar aynı `df` tablosuna eklenir.


In [3]:
result = build_basic_features(df)
df = result.df

display(pd.DataFrame({"metric": ["row_count", "total_columns", "new_feature_count"],
                      "value": [len(df), df.shape[1], len(result.feature_dictionary)]}))

,metric,value
0,row_count,192
1,total_columns,34
2,new_feature_count,14


### 4. Özellik sözlüğü ve hesaplama parametreleri

Her yeni kolonun tanımı, eksik değer davranışı ve yorumlama uyarısı açıkça kaydedilir. Ağırlıklı Google puanındaki iki parametre veriden hesaplanır ve ayrıca raporlanır.


In [4]:
display(result.feature_dictionary)
display(result.parameters)

,feature,dtype,feature_group,definition,missing_policy,caution
0,has_price_snapshot,boolean,availability,Fiyat snapshot değerinin bulunup bulunmadığı.,Eksik fiyat doldurulmaz; False olarak işaretle...,Fiyatın kalitesi veya güncelliği hakkında hükü...
1,has_official_star_rating,boolean,availability,Doğrulanmış resmî yıldız bilgisinin bulunup bu...,Eksik yıldız doldurulmaz; False olarak işaretl...,Google müşteri puanından yıldız türetilmez.
2,has_phone,boolean,availability,Telefon bilgisinin bulunup bulunmadığı.,Eksik telefon doldurulmaz; False olarak işaret...,İletişim kalitesini ölçmez.
3,has_business_status,boolean,availability,Google business status bilgisinin bulunup bulu...,Eksik durum doldurulmaz; False olarak işaretle...,Eksiklik işletmenin kapalı olduğu anlamına gel...
4,review_count_log1p,Float64,distribution,Google yorum sayısının log(1+x) dönüşümü.,Yorum sayısı eksikse sonuç eksik kalır.,Orijinal yorum sayısının yerine geçmez.
5,review_confidence_weight,Float64,rating,"v/(v+m); v yorum sayısı, m veri seti medyan yo...",Puan veya yorum sayısı eksikse sonuç eksik kalır.,İstatistiksel güven aralığı değildir; şeffaf b...
6,weighted_google_rating,Float64,rating,Yorum hacmine göre genel ortalamaya daraltılmı...,Puan veya yorum sayısı eksikse sonuç eksik kalır.,Rating tahmininde girdi olarak kullanılırsa he...
7,area_median_google_rating,Float64,area_context,Otelin bölgesindeki medyan Google müşteri puanı.,Bölgedeki mevcut puanlardan hesaplanır.,Küçük bölgelerde örneklem hassasiyeti vardır.
8,rating_gap_from_area_median,Float64,area_context,Otel puanı eksi bölge medyan puanı.,Gerekli değer eksikse sonuç eksik kalır.,Nedensel etki göstermez.
9,area_median_price_snapshot,Float64,price_context,Aynı bölgedeki dolu fiyat snapshotlarının medy...,Eksik fiyatlar medyan hesabında yok sayılır; d...,Yalnızca aynı snapshot bağlamında yorumlanmalı...


,parameter,value,definition
0,global_mean_google_rating,4.363542,Ağırlıklı puanda kullanılan tüm otellerin orta...
1,review_count_median_m,466.000000,Ağırlıklı puanda kullanılan veri seti medyan y...


### 5. Yorum hacmi ve ağırlıklı puan

Yorum sayısı sağa çarpık olduğundan `log(1+x)` dönüşümü hazırlanır. Ağırlıklı puan formülü:

$$
WR = \frac{v}{v+m}R + \frac{m}{v+m}C
$$

Burada `R` otelin Google puanı, `v` yorum sayısı, `C` tüm otellerin ortalama Google puanı ve `m` medyan yorum sayısıdır.

Bu değer sıralama/EDA bağlamında kullanılabilir; doğrudan `google_rating` kullanılarak hesaplandığı için rating tahmin modelinde girdi yapılmamalıdır.


In [5]:
rating_feature_preview = df[
    [
        "hotel_id",
        "hotel_name",
        "google_rating",
        "google_review_count",
        "review_count_log1p",
        "review_confidence_weight",
        "weighted_google_rating",
    ]
].head(10)
display(rating_feature_preview)

,hotel_id,hotel_name,google_rating,google_review_count,review_count_log1p,review_confidence_weight,weighted_google_rating
0,BOD001,Aksoy Taş Ev,4.7,135,4.912655,0.224626,4.439119
1,BOD002,Armonia Holiday Village & Spa,4.2,3197,8.070281,0.872782,4.220805
2,BOD003,ASPAT HOTEL BODRUM Beach Restaurant,4.3,246,5.509388,0.345506,4.341588
3,BOD004,Bendis Beach Hotel,3.9,2469,7.811973,0.841227,3.973598
4,BOD005,ByAkkan,4.4,203,5.31812,0.303438,4.374605
5,BOD006,Club Marma Hotel,4.3,1373,7.225481,0.746601,4.316101
6,BOD007,Kefaluka Resort,4.4,4293,8.364974,0.90208,4.39643
7,BOD008,LA BREZZA SUITE &HOTEL,4.5,424,6.052089,0.476404,4.428551
8,BOD009,LİV OTEL,4.4,546,6.304449,0.539526,4.383212
9,BOD010,Mare Garden Hotel,4.2,288,5.666427,0.381963,4.301075


### 6. Bölgesel bağlam özellikleri

Puan ve fiyat karşılaştırmaları genel Bodrum ortalaması yerine otelin kendi bölgesindeki medyanı da dikkate alabilecek biçimde hazırlanır. Bunlar yalnızca göreli konum ölçüleridir; neden-sonuç veya fiyat/performans hükmü değildir.


In [6]:
area_feature_preview = df[
    [
        "hotel_id",
        "area",
        "area_hotel_count",
        "google_rating",
        "area_median_google_rating",
        "rating_gap_from_area_median",
        "search_price_usd_snapshot",
        "area_median_price_snapshot",
        "price_gap_from_area_median",
        "price_ratio_to_area_median",
        "price_percentile_within_area",
        "review_count_percentile_within_area",
    ]
].head(10)
display(area_feature_preview)

,hotel_id,area,area_hotel_count,google_rating,area_median_google_rating,rating_gap_from_area_median,search_price_usd_snapshot,area_median_price_snapshot,price_gap_from_area_median,price_ratio_to_area_median,price_percentile_within_area,review_count_percentile_within_area
0,BOD001,Akyarlar,16,4.7,4.4,0.3,194.0,192.5,1.5,1.007792,0.583333,0.125
1,BOD002,Akyarlar,16,4.2,4.4,-0.2,191.0,192.5,-1.5,0.992208,0.5,0.8125
2,BOD003,Akyarlar,16,4.3,4.4,-0.1,57.0,192.5,-135.5,0.296104,0.083333,0.25
3,BOD004,Akyarlar,16,3.9,4.4,-0.5,104.0,192.5,-88.5,0.54026,0.25,0.75
4,BOD005,Akyarlar,16,4.4,4.4,0.0,84.0,192.5,-108.5,0.436364,0.166667,0.1875
5,BOD006,Akyarlar,16,4.3,4.4,-0.1,NaN,192.5,<NA>,<NA>,<NA>,0.5625
6,BOD007,Akyarlar,16,4.4,4.4,0.0,NaN,192.5,<NA>,<NA>,<NA>,0.9375
7,BOD008,Akyarlar,16,4.5,4.4,0.1,121.0,192.5,-71.5,0.628571,0.333333,0.375
8,BOD009,Akyarlar,16,4.4,4.4,0.0,216.0,192.5,23.5,1.122078,0.666667,0.4375
9,BOD010,Akyarlar,16,4.2,4.4,-0.2,165.0,192.5,-27.5,0.857143,0.416667,0.3125


### 7. Eksik değer davranışı

Eksik fiyatlar doldurulmaz. Bu satırlarda otele ait fiyat farkı, fiyat oranı ve fiyat yüzdelik sırası da eksik kalır. Bölge medyanı, yalnızca aynı bölgedeki mevcut fiyat gözlemlerinden hesaplanır ve imputasyon amacıyla kullanılmaz.


In [7]:
missing_feature_report = pd.DataFrame(
    {
        "column": result.feature_dictionary["feature"],
        "missing_count": [int(df[column].isna().sum()) for column in result.feature_dictionary["feature"]],
        "missing_percentage": [float(df[column].isna().mean() * 100) for column in result.feature_dictionary["feature"]],
    }
).sort_values(["missing_count", "column"], ascending=[False, True], ignore_index=True)
display(missing_feature_report)

,column,missing_count,missing_percentage
0,price_gap_from_area_median,24,12.5
1,price_percentile_within_area,24,12.5
2,price_ratio_to_area_median,24,12.5
3,area_median_google_rating,0,0.0
4,area_median_price_snapshot,0,0.0
5,has_business_status,0,0.0
6,has_official_star_rating,0,0.0
7,has_phone,0,0.0
8,has_price_snapshot,0,0.0
9,rating_gap_from_area_median,0,0.0


### 8. Doğrulama kontrolleri

Kaynak kolonların değişmediği, anahtarların ve satırların korunduğu, özellik aralıklarının geçerli olduğu ve eksik fiyatların doldurulmadığı otomatik olarak test edilir.


In [8]:
display(result.validation_report)

failed_checks = result.validation_report.loc[result.validation_report["status"].eq("FAIL")]
assert failed_checks.empty, f'Başarısız feature kontrolleri:\n{failed_checks.to_string(index=False)}'

,check,status,issue_count,detail
0,row_count_preserved,PASS,0,Satır eklenmemeli veya silinmemelidir.
1,source_columns_unchanged,PASS,0,Değişen kaynak kolonlar: []
2,keys_preserved,PASS,0,Otel kimlikleri ve sırası korunmalıdır.
3,area_hotel_count_matches,PASS,0,Bölge sayısı ana tablodan hesaplanmalıdır.
4,confidence_weight_in_range,PASS,0,Güven ağırlığı 0-1 aralığında olmalıdır.
5,weighted_rating_in_range,PASS,0,Ağırlıklı puan 0-5 aralığında olmalıdır.
6,price_availability_matches,PASS,0,Fiyat varlık göstergesi gerçek eksiklikle eşle...
7,missing_prices_not_imputed,PASS,0,Eksik otel fiyatından otel düzeyi fiyat özelli...


### 9. Çıktıların kaydedilmesi

Zenginleştirilmiş tek tablo `data/processed/hotels_features.csv` olarak yazılır. Özellik sözlüğü, parametreler ve doğrulama raporu `reports/` altında saklanır.


In [9]:
output_paths = save_feature_outputs(result, PROCESSED_DIR, REPORTS_DIR)
display(pd.DataFrame([
    {"output": name, "path": str(path.relative_to(PROJECT_ROOT))}
    for name, path in output_paths.items()
]))

,output,path
0,feature_table,data/processed/hotels_features.csv
1,feature_dictionary,reports/feature_dictionary.csv
2,parameters,reports/feature_engineering_parameters.csv
3,validation_report,reports/feature_validation_report.csv


### 10. Sonuç

- 192 otel satırı ve tüm kaynak kolonlar değişmeden korunmuştur.
- Aynı `df` içine 14 temel özellik eklenmiştir.
- Eksik fiyat ve resmî yıldız bilgileri doldurulmamıştır.
- Ayrı bölge özet tablosu kullanılmamıştır.
- Bileşik değer, lüks ve destinasyon skorları EDA sonrasına bırakılmıştır.
- Sonraki aşama `05_general_hotel_eda.ipynb` içinde hem kaynak değişkenlerin hem de bu temel özelliklerin dağılım ve ilişkilerini yorumlamaktır.
